# ============================================================
# Author: Mayur Deshmukh
# Title: 03_feature_engineer.ipynb
# Project: ML-Binary-Classifier-For-Stock-Price-Prediction
# Purpose: Feature Engineering for Stock Up/Down Prediction Model
# Python Version: 3.11
# ============================================================

In [1]:
import pandas as pd
import numpy as np
import os
import ta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

output_dir = os.path.join('..', '..', 'output')

In [2]:
# ── 1. Load EDA Dataset ─────────────────────────────────────────────────────────

df = pd.read_csv(os.path.join(output_dir, 'eda_stocks.csv'), parse_dates=['date'])
df = df.sort_values(['Name', 'date']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Tickers: {df['Name'].unique().tolist()}")
df.head()

Shape: (12590, 8)
Date range: 2013-02-08 → 2018-02-07
Tickers: ['A', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABT', 'ACN', 'WMT', 'ZTS']


,date,open,high,low,close,volume,Name,target
0,2013-02-08,45.07,45.35,45.00,45.08,1824755,A,0
1,2013-02-11,45.17,45.18,44.45,44.60,2915405,A,0
2,2013-02-12,44.81,44.95,44.50,44.62,2373731,A,1
3,2013-02-13,44.81,45.24,44.68,44.75,2052338,A,1
4,2013-02-14,44.72,44.78,44.36,44.58,3826245,A,0


In [3]:
# ── 2. Technical Indicators via `ta` library ────────────────────────────────────
# All indicators are computed per ticker to avoid cross-ticker contamination

def add_indicators(grp):
    close  = grp['close']
    high   = grp['high']
    low    = grp['low']
    volume = grp['volume']

    # SMA
    grp['sma_20'] = ta.trend.sma_indicator(close, window=20)
    grp['sma_50'] = ta.trend.sma_indicator(close, window=50)

    # EMA
    grp['ema_12'] = ta.trend.ema_indicator(close, window=12)
    grp['ema_26'] = ta.trend.ema_indicator(close, window=26)

    # MACD
    macd_obj          = ta.trend.MACD(close, window_slow=26, window_fast=12, window_sign=9)
    grp['macd_line']  = macd_obj.macd()
    grp['macd_signal']= macd_obj.macd_signal()
    grp['macd_hist']  = macd_obj.macd_diff()

    # RSI (14)
    grp['rsi_14'] = ta.momentum.rsi(close, window=14)

    # Bollinger Bands
    bb               = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    grp['bb_upper']  = bb.bollinger_hband()
    grp['bb_middle'] = bb.bollinger_mavg()
    grp['bb_lower']  = bb.bollinger_lband()
    grp['bb_pct_b']  = bb.bollinger_pband()   # %B

    return grp

df = df.groupby('Name', group_keys=False).apply(add_indicators)

# Drop rows where any indicator is NaN (warm-up period of slowest indicator = SMA-50)
df = df.dropna().reset_index(drop=True)

print(f"Shape after adding indicators: {df.shape}")
print(f"New columns: {[c for c in df.columns if c not in ['date','open','high','low','close','volume','Name','target']]}")
df.head()

Shape after adding indicators: (12100, 19)
New columns: ['sma_20', 'sma_50', 'ema_12', 'ema_26', 'macd_line', 'macd_signal', 'macd_hist', 'rsi_14', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_pct_b']


,date,open,high,low,close,volume,target,sma_20,sma_50,ema_12,ema_26,macd_line,macd_signal,macd_hist,rsi_14,bb_upper,bb_middle,bb_lower,bb_pct_b
0,2013-04-22,42.03,42.2500,41.34,41.85,2340457,0,42.0275,42.4326,42.310040,42.320150,-0.010110,-0.018301,0.008190,46.565393,44.072141,42.0275,39.982859,0.456594
1,2013-04-23,42.21,42.8425,42.13,42.61,3522419,1,42.1015,42.3832,42.356188,42.341620,0.014567,-0.011727,0.026294,51.815841,44.117786,42.1015,40.085214,0.626098
2,2013-04-24,42.72,43.0350,42.37,42.85,3020692,1,42.1530,42.3482,42.432159,42.379278,0.052881,0.001195,0.051686,53.373904,44.190401,42.1530,40.115599,0.671051
3,2013-04-25,43.05,43.2500,42.66,42.73,2119838,0,42.1850,42.3104,42.477980,42.405258,0.072723,0.015500,0.057223,52.460490,44.237486,42.1850,40.132514,0.632766
4,2013-04-26,42.44,42.5700,40.69,41.30,5283798,0,42.1515,42.2414,42.296753,42.323387,-0.026634,0.007073,-0.033707,43.013684,44.238510,42.1515,40.064490,0.296000


In [4]:
# ── 3. Create Feature Matrix (X) and Response Vector (y) ────────────────────────

feature_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'sma_20', 'sma_50', 'ema_12', 'ema_26',
    'macd_line', 'macd_signal', 'macd_hist',
    'rsi_14',
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_pct_b'
]

X = df[feature_cols].copy()
y = df['target'].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny value counts:\n{y.value_counts()}")

X shape: (12100, 17)
y shape: (12100,)

y value counts:
target
1    6318
0    5782
Name: count, dtype: int64


In [5]:
# ── 4. Chronological Train-Test Split ───────────────────────────────────────────
# Stock data is temporal — we must split by date, not randomly.
# Strategy: snap the cut-off to a full year boundary closest to an 80/20 split.

years = sorted(df['date'].dt.year.unique())
total = len(df)

# Find the year-end cut-off that gives the closest split to 80/20
best_year, best_diff = None, float('inf')
for yr in years[:-1]:                          # keep at least one year in test
    n_train = (df['date'].dt.year <= yr).sum()
    diff = abs(n_train / total - 0.80)
    if diff < best_diff:
        best_diff, best_year = diff, yr

cutoff_date = pd.Timestamp(f'{best_year}-12-31')

train_mask = df['date'] <= cutoff_date
X_train, X_test = X[train_mask].copy(), X[~train_mask].copy()
y_train, y_test = y[train_mask].copy(), y[~train_mask].copy()

print(f"Cut-off date  : {cutoff_date.date()}")
print(f"Train period  : {df.loc[train_mask,  'date'].min().date()} → {df.loc[train_mask,  'date'].max().date()}")
print(f"Test  period  : {df.loc[~train_mask, 'date'].min().date()} → {df.loc[~train_mask, 'date'].max().date()}")
print(f"\nX_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")
print(f"Actual split  : {len(X_train)/total:.1%} train / {len(X_test)/total:.1%} test")

Cut-off date  : 2016-12-31
Train period  : 2013-04-22 → 2016-12-30
Test  period  : 2017-01-03 → 2018-02-07

X_train: (9330, 17)  |  X_test: (2770, 17)
y_train: (9330,)  |  y_test: (2770,)
Actual split  : 77.1% train / 22.9% test


In [6]:
# ── 5. MinMax Scaling on Features ───────────────────────────────────────────────
# Fit scaler ONLY on X_train, then transform both X_train and X_test.
# Fitting on X_test would cause data leakage.

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols, index=X_test.index)

print("X_train_scaled — min/max check (should be 0.0 / 1.0 for all cols):")
print(X_train_scaled.agg(['min', 'max']).T.head())

print("\nX_test_scaled — values may fall outside [0,1] due to unseen price ranges:")
print(X_test_scaled.agg(['min', 'max']).T.head())

X_train_scaled — min/max check (should be 0.0 / 1.0 for all cols):
        min  max
open    0.0  1.0
high    0.0  1.0
low     0.0  1.0
close   0.0  1.0
volume  0.0  1.0

X_test_scaled — values may fall outside [0,1] due to unseen price ranges:
             min       max
open    0.125348  0.882362
high    0.126811  0.886167
low     0.126094  0.891131
close   0.128134  0.885863
volume  0.000949  0.419397


In [7]:
# ── 6. Label Encoding for y_train / y_test ──────────────────────────────────────
# target is already binary integer (0 = Down, 1 = Up) — no encoding needed.
# sklearn classifiers accept integer labels directly.
# LabelEncoder would map {0→0, 1→1} — a no-op — so we skip it intentionally.

print("y_train dtype :", y_train.dtype)
print("y_test  dtype :", y_test.dtype)
print(f"\ny_train distribution:\n{y_train.value_counts()}")
print(f"\ny_test  distribution:\n{y_test.value_counts()}")

y_train dtype : int64
y_test  dtype : int64

y_train distribution:
target
1    4818
0    4512
Name: count, dtype: int64

y_test  distribution:
target
1    1500
0    1270
Name: count, dtype: int64


In [8]:
# ── 7. Export Feature-Engineered Datasets ───────────────────────────────────────

X_train_scaled.to_csv(os.path.join(output_dir, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(output_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(output_dir, 'y_train.csv'), index=False, header=True)
y_test.to_csv(os.path.join(output_dir, 'y_test.csv'), index=False, header=True)

print("Exported:")
print(f"  X_train.csv : {X_train_scaled.shape}")
print(f"  X_test.csv  : {X_test_scaled.shape}")
print(f"  y_train.csv : {y_train.shape}")
print(f"  y_test.csv  : {y_test.shape}")

Exported:
  X_train.csv : (9330, 17)
  X_test.csv  : (2770, 17)
  y_train.csv : (9330,)
  y_test.csv  : (2770,)
